<a href="https://colab.research.google.com/github/Jeremylaby/Podstawy-SI/blob/lab3_sieci_neuronowe/lab3/lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sieci neuronowe

## Wst?p

Celem laboratorium jest zapoznanie si? z podstawami sieci neuronowych oraz uczeniem g??bokim (*deep learning*). Zapoznasz si? na nim z nast?puj?cymi tematami:
- treningiem prostych sieci neuronowych, w szczeg?lno?ci z:
  - regresj? liniow? w sieciach neuronowych
  - optymalizacj? funkcji kosztu
  - algorytmem spadku wzd?u? gradientu
  - sieci? typu Multilayer Perceptron (MLP)
- frameworkiem PyTorch, w szczeg?lno?ci z:
  - ?adowaniem danych
  - preprocessingiem danych
  - pisaniem p?tli treningowej i walidacyjnej
  - walidacj? modeli
- architektur? i hiperaprametrami sieci MLP, w szczeg?lno?ci z:
  - warstwami g?stymi (w pe?ni po??czonymi)
  - funkcjami aktywacji
  - regularyzacj?: L2, dropout

## Wykorzystywane biblioteki

Zaczniemy od pisania r?cznie prostych sieci w bibliotece Numpy, s?u??cej do oblicze? numerycznych na CPU. P??niej przejdziemy do wykorzystywania frameworka PyTorch, s?u??cego do oblicze? numerycznych na CPU, GPU oraz automatycznego r??niczkowania, wykorzystywanego g??wnie do treningu sieci neuronowych.

Wykorzystamy PyTorcha ze wzgl?du na popularno??, ?atwo?? instalacji i u?ycia, oraz du?? kontrol? nad niskopoziomowymi aspektami budowy i treningu sieci neuronowych. Framework ten zosta? stworzony do zastosowa? badawczych i naukowych, ale ze wzgl?du na wygod? u?ycia sta? si? bardzo popularny tak?e w przemy?le. W szczeg?lno?ci ca?kowicie zdominowa? przetwarzanie j?zyka naturalnego (NLP) oraz uczenie na grafach.

Pierwszy du?y framework do deep learningu, oraz obecnie najpopularniejszy, to TensorFlow, wraz z wysokopoziomow? nak?adk? Keras. S? jednak szanse, ?e Google (autorzy) b?dzie go powoli porzuca? na rzecz ich nowego frameworka JAX ([dyskusja](https://www.reddit.com/r/MachineLearning/comments/vfl57t/d_google_quietly_moving_its_products_from/), [artyku? Business Insidera](https://www.businessinsider.com/facebook-pytorch-beat-google-tensorflow-jax-meta-ai-2022-6?IR=T)), kt?ry jest bardzo ?wie?ym, ale ciekawym narz?dziem.

Trzecia, ale znacznie mniej popularna od powy?szych opcja to Apache MXNet.

## Konfiguracja w?asnego komputera

Je?eli korzystasz z w?asnego komputera, to musisz zainstalowa? troch? wi?cej bibliotek (Google Colab ma je ju? zainstalowane).

Je?eli nie masz GPU lub nie chcesz z niego korzysta?, to wystarczy znale?? odpowiedni? komend? CPU [na stronie PyTorcha](https://pytorch.org/get-started/locally/). Dla Anacondy odpowiednia komenda zosta?a podana poni?ej, dla pip'a znajd? j? na stronie.

Je?eli chcesz korzysta? ze wsparcia GPU (na tym laboratorium nie b?dzie potrzebne, na kolejnych mo?e przyspieszy? nieco obliczenia), to musi by? to odpowiednio nowa karta NVidii, maj?ca CUDA compatibility ([lista](https://developer.nvidia.com/cuda-gpus)). Poza PyTorchem b?dzie potrzebne narz?dzie NVidia CUDA w wersji 11.6 lub 11.7. Instalacja na Windowsie jest bardzo prosta (wystarczy ?ci?gn?? plik EXE i zainstalowa? jak ka?dy inny program). Instalacja na Linuxie jest trudna i mo?na wzgl?dnie ?atwo zepsu? sobie system, ale je?eli chcesz spr?bowa?, to [ten tutorial](https://www.youtube.com/results?search_query=nvidia+cuda+install+ubuntu+20.04) jest bardzo dobry.

## Wprowadzenie

Zanim zaczniemy nasz? przygod? z sieciami neuronowymi, przyjrzyjmy si? prostemu przyk?adowi regresji liniowej na syntetycznych danych:

In [ ]:
from typing import Tuple, Dict

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(0)

x = np.linspace(0, 1, 100)
y = x + np.random.normal(scale=0.1, size=x.shape)

plt.scatter(x, y)

W przeciwie?stwie do laboratorium 1, tym razem b?dziemy chcieli rozwi?za? ten problem w?asnor?cznie, bez u?ycia wysokopoziomowego interfejsu Scikit-learn'a. W tym celu musimy sobie przypomnie? sformu?owanie naszego **problemu optymalizacyjnego (optimization problem)**.

W przypadku prostej regresji liniowej (1 zmienna) mamy model postaci $\hat{y} = \alpha x + \beta$, z dwoma parametrami, kt?rych b?dziemy si? uczy?. Miar? niedopasowania modelu o danych parametrach jest **funkcja kosztu (cost function)**, nazywana te? funkcj? celu. Najcz??ciej u?ywa si? **b??du ?redniokwadratowego (mean squared error, MSE)**:
$$\large
MSE = \frac{1}{N} \sum_{i}^{N} (y - \hat{y})^2
$$

Od jakich $\alpha$ i $\beta$ zacz??? W najprostszym wypadku wystarczy po prostu je wylosowa? jako niewielkie liczby zmiennoprzecinkowe.

### Zadanie 1 (0.5 punkt)

Uzupe?nij kod funkcji `mse`, obliczaj?cej b??d ?redniokwadratowy. Wykorzystaj Numpy'a w celu wektoryzacji oblicze? dla wydajno?ci.

In [ ]:
def mse(y: np.ndarray, y_hat: np.ndarray) -> float:
    # implement me!
    # your_code
    return np.mean((y - y_hat) ** 2)



In [ ]:
a = np.random.rand()
b = np.random.rand()
print(f"MSE: {mse(y, a * x + b):.3f}")

plt.scatter(x, y)
plt.plot(x, a * x + b, color="g", linewidth=4)

Losowe parametry radz? sobie nie najlepiej. Jak lepiej dopasowa? nasz? prost? do danych? Zawsze mo?emy stara? si? wyprowadzi? rozwi?zanie analitycznie, i w tym wypadku nawet nam si? uda. Jest to jednak szczeg?lny i do?? rzadki przypadek, a w szczeg?lno?ci nie b?dzie to mo?liwe w wi?kszych sieciach neuronowych.

Potrzebna nam b?dzie **metoda optymalizacji (optimization method)**, daj?c? warto?ci parametr?w minimalizuj?ce dowoln? r??niczkowaln? funkcj? kosztu. Zdecydowanie najpopularniejszy jest tutaj **spadek wzd?u? gradientu (gradient descent)**.

Metoda ta wywodzi si? z prostych obserwacji, kt?re tutaj przedstawimy. Bardziej szczeg??owe rozwini?cie dla zainteresowanych: [sekcja 4.3 "Deep Learning Book"](https://www.deeplearningbook.org/contents/numerical.html), [ten praktyczny kurs](https://cs231n.github.io/optimization-1/), [analiza oryginalnej publikacji Cauchy'ego](https://www.math.uni-bielefeld.de/documenta/vol-ismp/40_lemarechal-claude.pdf) (orygina? w j?zyku francuskim).

Pochodna jest dok?adnie r?wna granicy funkcji. Dla ma?ego $\epsilon$ mo?na j? przybli?y? jako:
$$\large
\frac{f(x)}{dx} \approx \frac{f(x+\epsilon) - f(x)}{\epsilon}
$$

Przygl?daj?c si? temu r?wnaniu widzimy, ?e:
* dla funkcji rosn?cej ($f(x+\epsilon) > f(x)$) wyra?enie $\frac{f(x)}{dx}$ b?dzie mia?o znak dodatni
* dla funkcji malej?cej ($f(x+\epsilon) < f(x)$) wyra?enie $\frac{f(x)}{dx}$ b?dzie mia?o znak ujemny

Widzimy wi?c, ?e potrafimy wskaza? kierunek zmniejszenia warto?ci funkcji, patrz?c na znak pochodnej. Zaobserwowano tak?e, ?e amplituda warto?ci w $\frac{f(x)}{dx}$ jest tym wi?ksza, im dalej jeste?my od minimum (maximum). Pochodna wyznacza wi?c, w jakim kierunku funkcja najszybciej ro?nie, za? przeciwny zwrot to ten, w kt?rym funkcja najszybciej spada.

Stosuj?c powy?sze do optymalizacji, mamy:
$$\large
x_{t+1} = x_{t} -  \alpha * \frac{f(x)}{dx}
$$

$\alpha$ to niewielka warto?? (rz?du zwykle $10^{-5}$ - $10^{-2}$), wprowadzona, aby trzyma? si? za?o?enia o ma?ej zmianie parametr?w ($\epsilon$). Nazywa si? j? **sta?? ucz?c? (learning rate)** i jest zwykle najwa?niejszym hiperparametrem podczas nauki sieci.

Metoda ta zak?ada, ?e u?ywamy ca?ego zbioru danych do aktualizacji parametr?w w ka?dym kroku, co nazywa si? po prostu GD (od *gradient descent*) albo *full batch GD*. Wtedy ka?dy krok optymalizacji nazywa si? **epok? (epoch)**.

Im wi?ksza sta?a ucz?ca, tym wi?ksze nasze kroki podczas minimalizacji. Mo?emy wi?c uczy? szybciej, ale istnieje ryzyko, ?e b?dziemy "przeskakiwa?" minima. Mniejsza sta?a ucz?ca to wolniejszy, ale dok?adniejszy trening. Jednak nie zawsze ona pozwala osi?gn?? lepsze wyniki, bo mo?e okaza? si?, ?e utkniemy w minimum lokalnym. Mo?na tak?e zmienia? sta?? ucz?c? podczas treningu, co nazywa si? **learning rate scheduling (LR scheduling)**. Obrazowo:

![learning_rate](http://www.bdhammel.com/assets/learning-rate/lr-types.png)

![interactive LR](http://cdn-images-1.medium.com/max/640/1*eeIvlwkMNG1wSmj3FR6M2g.gif)

Policzmy wi?c pochodn? dla naszej funkcji kosztu MSE. Pochodn? liczymy po parametrach naszego modelu, bo to w?a?nie ich chcemy dopasowa? tak, ?eby koszt by? jak najmniejszy:

$$\large
MSE = \frac{1}{N} \sum_{i}^{N} (y_i - \hat{y_i})^2
$$

W powy?szym wzorze tylko $y_i$ jest zale?ny od $a$ oraz $b$. Mo?emy wykorzysta? tu regu?? ?a?cuchow? (*chain rule*) i policzy? pochodne po naszych parametrach w spos?b nast?puj?cy:

$$\large
\frac{\text{d} MSE}{\text{d} a} = \frac{1}{N} \sum_{i}^{N} \frac{\text{d} (y_i - \hat{y_i})^2}{\text{d} \hat{y_i}} \frac{\text{d} \hat{y_i}}{\text{d} a}
$$

$$\large
\frac{\text{d} MSE}{\text{d} b} = \frac{1}{N} \sum_{i}^{N} \frac{\text{d} (y_i - \hat{y_i})^2}{\text{d} \hat{y_i}} \frac{\text{d} \hat{y_i}}{\text{d} b}
$$

Policzmy te pochodne po kolei:

$$\large
\frac{\text{d} (y_i - \hat{y_i})^2}{\text{d} \hat{y_i}} = -2 \cdot (y_i - \hat{y_i})
$$

$$\large
\frac{\text{d} \hat{y_i}}{\text{d} a} = x_i
$$

$$\large
\frac{\text{d} \hat{y_i}}{\text{d} b} = 1
$$

??cz?c powy?sze wyniki dostaniemy:

$$\large
\frac{\text{d} MSE}{\text{d} a} = \frac{-2}{N} \sum_{i}^{N} (y_i - \hat{y_i}) \cdot {x_i}
$$

$$\large
\frac{\text{d} MSE}{\text{d} b} = \frac{-2}{N} \sum_{i}^{N} (y_i - \hat{y_i})
$$

Aktualizacja parametr?w wygl?da tak:

$$\large
a' = a - \alpha * \left( \frac{-2}{N} \sum_{i=1}^N (y_i - \hat{y}_i) \cdot x_i \right)
$$
$$\large
b' = b - \alpha * \left( \frac{-2}{N} \sum_{i=1}^N (y_i - \hat{y}_i) \right)
$$

Liczymy wi?c pochodn? funkcji kosztu, a potem za pomoc? regu?y ?a?cuchowej "cofamy si?", dochodz?c do tego, jak ka?dy z parametr?w wp?ywa na b??d i w jaki spos?b powinni?my go zmieni?. Nazywa si? to **propagacj? wsteczn? (backpropagation)** i jest podstawowym mechanizmem umo?liwiaj?cym nauk? sieci neuronowych za pomoc? spadku wzd?u? gradientu. Wi?cej mo?esz o tym przeczyta? [tutaj](https://cs231n.github.io/optimization-2/).

### Zadanie 2 (1.0 punkt)

Zaimplementuj funkcj? realizuj?c? jedn? epok? treningow?. Zauwa?, ?e `x` oraz `y` s? wektorami. Oblicz predykcj? przy aktualnych parametrach oraz zaktualizuj je zgodnie z powy?szymi wzorami.

In [ ]:
def optimize(
    x: np.ndarray, y: np.ndarray, a: float, b: float, learning_rate: float = 0.1
):
    y_hat = a * x + b
    errors = y - y_hat
    # implement me!
    # your_code
    a = a - learning_rate *-2* (errors * x).mean()
    b = b - learning_rate *-2* errors.mean()
    return a, b


In [ ]:
for i in range(1000):
    loss = mse(y, a * x + b)
    a, b = optimize(x, y, a, b)
    if i % 100 == 0:
        print(f"step {i} loss: ", loss)

print("final loss:", loss)

In [ ]:
assert 0.0 < loss < 2.0

print("Solution is correct!")

In [ ]:
plt.scatter(x, y)
plt.plot(x, a * x + b, color="g", linewidth=4)

Uda?o ci si? wytrenowa? swoj? pierwsz? sie? neuronow?. Czemu? Ot?? neuron to po prostu wektor parametr?w, a zwykle robimy iloczyn skalarny tych parametr?w z wej?ciem. Dodatkowo na wyj?cie nak?ada si? **funkcj? aktywacji (activation function)**, kt?ra przekszta?ca wyj?cie. Tutaj takiej nie by?o, a w?a?ciwie by?a to po prostu funkcja identyczno?ci.

Oczywi?cie w praktyce korzystamy z odpowiedniego frameworka, kt?ry w szczeg?lno?ci:
- u?atwia budowanie sieci, np. ma gotowe klasy dla warstw neuron?w
- ma zaimplementowane funkcje kosztu oraz ich pochodne
- sam r??niczkuje ze wzgl?du na odpowiednie parametry i aktualizuje je odpowiednio podczas treningu


## Wprowadzenie do PyTorcha

PyTorch to w gruncie rzeczy narz?dzie do algebry liniowej z [automatycznym ro?niczkowaniem](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html), z mo?liwo?ci? przyspieszenia oblicze? z pomoc? GPU. Na tych fundamentach zbudowany jest pe?ny framework do uczenia g??bokiego. Mo?na spotka? si? ze stwierdzenie, ?e PyTorch to NumPy + GPU + opcjonalne r??niczkowanie, co jest ca?kiem celne. Plus mo?na ?atwo debugowa? printem :)

PyTorch u?ywa dynamicznego grafu oblicze?, kt?ry sami definiujemy w kodzie. Takie podej?cie jest bardzo wygodne, elastyczne i pozwala na ?atwe eksperymentowanie. Odbywa si? to potencjalnie kosztem wydajno?ci, poniewa? pozostawia kwesti? optymalizacji programi?cie. Wi?cej na ten temat dla zainteresowanych na ko?cu laboratorium.

Samo API PyTorcha bardzo przypomina Numpy'a, a podstawowym obiektem jest `Tensor`, klasa reprezentuj?ca tensory dowolnego wymiaru. Dodatkowo niekt?re tensory b?d? mia?y automatycznie obliczony gradient. Co wa?ne, tensor jest na pewnym urz?dzeniu, CPU lub GPU, a przenosi? mi?dzy nimi trzeba explicite.

Najwa?niejsze modu?y:
- `torch` - podstawowe klasy oraz funkcje, np. `Tensor`, `from_numpy()`
- `torch.nn` - klasy zwi?zane z sieciami neuronowymi, np. `Linear`, `Sigmoid`
- `torch.optim` - wszystko zwi?zane z optymalizacj?, g??wnie spadkiem wzd?u? gradientu

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
ones = torch.ones(10)
noise = torch.ones(10) * torch.rand(10)

# elementwise sum
print(ones + noise)

# elementwise multiplication
print(ones * noise)

# dot product
print(ones @ noise)

In [ ]:
type(x)

In [ ]:
# beware - shares memory with original Numpy array!
# very fast, but modifications are visible to original variable
x = torch.from_numpy(x)
y = torch.from_numpy(y)

Je?eli dla stworzonych przez nas tensor?w chcemy ?ledzi? operacje i oblicza? gradient, to musimy oznaczy? `requires_grad=True`.

In [ ]:
a = torch.rand(1, requires_grad=True)
b = torch.rand(1, requires_grad=True)
a, b

PyTorch zawiera wi?kszo?? powszechnie u?ywanych funkcji kosztu, np. MSE. Mog? by? one u?ywane na 2 sposoby, z czego pierwszy jest popularniejszy:
- jako klasy wywo?ywalne z modu?u `torch.nn`
- jako funkcje z modu?u `torch.nn.functional`

Po wykonaniu poni?szego kodu widzimy, ?e zwraca on nam tensor z dodatkowymi atrybutami. Co wa?ne, jest to skalar (0-wymiarowy tensor), bo potrzebujemy zwyczajnej liczby do obliczania propagacji wstecznych (pochodnych cz?tkowych).

In [ ]:
mse = nn.MSELoss()
mse(y, a * x + b)

Atrybutu `grad_fn` nie u?ywamy wprost, bo korzysta z niego w ?rodku PyTorch, ale wida?, ?e tensor jest "?wiadomy", ?e liczy si? na nim pochodn?. Mo?emy natomiast skorzysta? z atrybutu `grad`, kt?ry zawiera faktyczny gradient. Zanim go jednak dostaniemy, to trzeba powiedzie? PyTorchowi, ?eby policzy? gradient. S?u?y do tego metoda `.backward()`, wywo?ywana na obiekcie zwracanym przez funkcj? kosztu.

In [ ]:
loss = mse(y, a * x + b)
loss.backward()

In [ ]:
print(a.grad)

Wa?ne jest, ?e PyTorch nie liczy za ka?dym razem nowego gradientu, tylko dodaje go do istniej?cego, czyli go akumuluje. Jest to przydatne w niekt?rych sieciach neuronowych, ale zazwyczaj trzeba go zerowa?. Je?eli tego nie zrobimy, to dostaniemy coraz wi?ksze gradienty.

Do zerowania s?u?y metoda `.zero_()`. W PyTorchu wszystkie metody modyfikuj?ce tensor w miejscu maj? `_` na ko?cu nazwy. Jest to do?? niskopoziomowa operacja dla pojedynczych tensor?w - zobaczymy za chwil?, jak to robi? ?atwiej dla ca?ej sieci.

In [ ]:
loss = mse(y, a * x + b)
loss.backward()
a.grad

Zobaczmy, jak wygl?da?aby regresja liniowa, ale napisana w PyTorchu. Jest to oczywi?cie bardzo niskopoziomowa implementacja - za chwil? zobaczymy, jak to wygl?da w praktyce.

In [ ]:
learning_rate = 0.1
for i in range(1000):
    loss = mse(y, a * x + b)

    # compute gradients
    loss.backward()

    # update parameters
    a.data -= learning_rate * a.grad
    b.data -= learning_rate * b.grad

    # zero gradients
    a.grad.data.zero_()
    b.grad.data.zero_()

    if i % 100 == 0:
        print(f"step {i} loss: ", loss)

print("final loss:", loss)

Trening modeli w PyTorchu jest dosy? schematyczny i najcz??ciej rozdziela si? go na kilka blok?w, daj?cych razem **p?tl? ucz?c? (training loop)**, powtarzan? w ka?dej epoce:
1. Forward pass - obliczenie predykcji sieci
2. Loss calculation
3. Backpropagation - obliczenie pochodnych oraz zerowanie gradient?w
4. Optimalization - aktualizacja wag
5. Other - ewaluacja na zbiorze walidacyjnym, logging etc.

In [ ]:
# initialization
learning_rate = 0.1
a = torch.rand(1, requires_grad=True)
b = torch.rand(1, requires_grad=True)
optimizer = torch.optim.SGD([a, b], lr=learning_rate)
best_loss = float("inf")

# training loop in each epoch
for i in range(1000):
    # forward pass
    y_hat = a * x + b

    # loss calculation
    loss = mse(y, y_hat)

    # backpropagation
    loss.backward()

    # optimization
    optimizer.step()
    optimizer.zero_grad()  # zeroes all gradients - very convenient!

    if i % 100 == 0:
        if loss < best_loss:
            best_model = (a.clone(), b.clone())
            best_loss = loss
        print(f"step {i} loss: {loss.item():.4f}")

print("final loss:", loss)

Przejdziemy teraz do budowy sieci neuronowej do klasyfikacji. Typowo implementuje si? j? po prostu jako sie? dla regresji, ale zwracaj?c? tyle wynik?w, ile mamy klas, a potem aplikuje si? na tym funkcj? sigmoidaln? (2 klasy) lub softmax (>2 klasy). W przypadku klasyfikacji binarnej zwraca si? czasem tylko 1 warto??, przepuszczan? przez sigmoid? - wtedy wyj?cie z sieci to prawdopodobie?stwo klasy pozytywnej.

Funkcj? kosztu zwykle jest **entropia krzy?owa (cross-entropy)**, stosowana te? w klasycznej regresji logistycznej. Co wa?ne, sieci neuronowe, nawet tak proste, ucz? si? szybciej i stabilniej, gdy dane na wej?ciu (a przynajmniej zmienne numeryczne) s? **ustandaryzowane (standardized)**. Operacja ta polega na odj?ciu ?redniej i podzieleniu przez odchylenie standardowe (tzw. *Z-score transformation*).

**Uwaga - PyTorch wymaga tensora klas b?d?cego liczbami zmiennoprzecinkowymi!**

## Zbi?r danych

Na tym laboratorium wykorzystamy zbi?r [Adult Census](https://archive.ics.uci.edu/ml/datasets/adult). Dotyczy on przewidywania na podstawie danych demograficznych, czy dany cz?owiek zarabia powy?ej 50 tysi?cy dolar?w rocznie, czy te? mniej. Jest to cenna informacja np. przy planowaniu kampanii marketingowych. Jak mo?esz si? domy?li?, zbi?r pochodzi z czas?w, kiedy inflacja by?a du?o ni?sza :)

Poni?ej znajduje si? kod do ?ci?gni?cia i preprocessingu zbioru. Nie musisz go dok?adnie analizowa?.

In [ ]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data

In [ ]:
import pandas as pd


columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "wage"
]

"""
age: continuous.
workclass: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked.
fnlwgt: continuous.
education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool.
education-num: continuous.
marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse.
occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces.
relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried.
race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black.
sex: Female, Male.
capital-gain: continuous.
capital-loss: continuous.
hours-per-week: continuous.
native-country: United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands.
"""

df = pd.read_csv("adult.data", header=None, names=columns)
df.wage.unique()

In [ ]:
# attribution: https://www.kaggle.com/code/royshih23/topic7-classification-in-python
df['education'].replace('Preschool', 'dropout',inplace=True)
df['education'].replace('10th', 'dropout',inplace=True)
df['education'].replace('11th', 'dropout',inplace=True)
df['education'].replace('12th', 'dropout',inplace=True)
df['education'].replace('1st-4th', 'dropout',inplace=True)
df['education'].replace('5th-6th', 'dropout',inplace=True)
df['education'].replace('7th-8th', 'dropout',inplace=True)
df['education'].replace('9th', 'dropout',inplace=True)
df['education'].replace('HS-Grad', 'HighGrad',inplace=True)
df['education'].replace('HS-grad', 'HighGrad',inplace=True)
df['education'].replace('Some-college', 'CommunityCollege',inplace=True)
df['education'].replace('Assoc-acdm', 'CommunityCollege',inplace=True)
df['education'].replace('Assoc-voc', 'CommunityCollege',inplace=True)
df['education'].replace('Bachelors', 'Bachelors',inplace=True)
df['education'].replace('Masters', 'Masters',inplace=True)
df['education'].replace('Prof-school', 'Masters',inplace=True)
df['education'].replace('Doctorate', 'Doctorate',inplace=True)

df['marital-status'].replace('Never-married', 'NotMarried',inplace=True)
df['marital-status'].replace(['Married-AF-spouse'], 'Married',inplace=True)
df['marital-status'].replace(['Married-civ-spouse'], 'Married',inplace=True)
df['marital-status'].replace(['Married-spouse-absent'], 'NotMarried',inplace=True)
df['marital-status'].replace(['Separated'], 'Separated',inplace=True)
df['marital-status'].replace(['Divorced'], 'Separated',inplace=True)
df['marital-status'].replace(['Widowed'], 'Widowed',inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler


X = df.copy()
y = (X.pop("wage") == ' >50K').astype(int).values

train_valid_size = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=train_valid_size,
    random_state=0,
    shuffle=True,
    stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train,
    test_size=train_valid_size,
    random_state=0,
    shuffle=True,
    stratify=y_train
)

continuous_cols = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']
continuous_X_train = X_train[continuous_cols]
categorical_X_train = X_train.loc[:, ~X_train.columns.isin(continuous_cols)]

continuous_X_valid = X_valid[continuous_cols]
categorical_X_valid = X_valid.loc[:, ~X_valid.columns.isin(continuous_cols)]

continuous_X_test = X_test[continuous_cols]
categorical_X_test = X_test.loc[:, ~X_test.columns.isin(continuous_cols)]

categorical_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
continuous_scaler = StandardScaler() #MinMaxScaler(feature_range=(-1, 1))

categorical_encoder.fit(categorical_X_train)
continuous_scaler.fit(continuous_X_train)

continuous_X_train = continuous_scaler.transform(continuous_X_train)
continuous_X_valid = continuous_scaler.transform(continuous_X_valid)
continuous_X_test = continuous_scaler.transform(continuous_X_test)

categorical_X_train = categorical_encoder.transform(categorical_X_train)
categorical_X_valid = categorical_encoder.transform(categorical_X_valid)
categorical_X_test = categorical_encoder.transform(categorical_X_test)

X_train = np.concatenate([continuous_X_train, categorical_X_train], axis=1)
X_valid = np.concatenate([continuous_X_valid, categorical_X_valid], axis=1)
X_test = np.concatenate([continuous_X_test, categorical_X_test], axis=1)

X_train.shape, y_train.shape

Uwaga co do typ?w - PyTorchu wszystko w sieci neuronowej musi by? typu `float32`. W szczeg?lno?ci trzeba uwa?a? na konwersje z Numpy'a, kt?ry u?ywa domy?lnie typu `float64`. Mo?e ci si? przyda? metoda `.float()`.

Uwaga co do kszta?t?w wyj?cia - wej?cia do `nn.BCELoss` musz? by? tego samego kszta?tu. Mo?e ci si? przyda? metoda `.squeeze()` lub `.unsqueeze()`.

In [ ]:
X_train = torch.from_numpy(X_train).float()
y_train = torch.from_numpy(y_train).float().unsqueeze(-1)

X_valid = torch.from_numpy(X_valid).float()
y_valid = torch.from_numpy(y_valid).float().unsqueeze(-1)

X_test = torch.from_numpy(X_test).float()
y_test = torch.from_numpy(y_test).float().unsqueeze(-1)

Podobnie jak w laboratorium 2, mamy tu do czynienia z klasyfikacj? niezbalansowan?:

In [ ]:
import matplotlib.pyplot as plt

y_pos_perc = 100 * y_train.sum().item() / len(y_train)
y_neg_perc = 100 - y_pos_perc

plt.title("Class percentages")
plt.bar(["<50k", ">=50k"], [y_neg_perc, y_pos_perc])
plt.show()

W zwi?zku z powy?szym b?dziemy u?ywa? odpowiednich metryk, czyli AUROC, precyzji i czu?o?ci.

### Zadanie 3 (1.0 punkt)

Zaimplementuj regresj? logistyczn? dla tego zbioru danych, u?ywaj?c PyTorcha. Dane wej?ciowe zosta?y dla ciebie przygotowane w kom?rkach poni?ej.

Sama sie? sk?ada si? z 2 element?w:
- warstwa liniowa `nn.Linear`, przekszta?caj?ca wektor wej?ciowy na 1 wyj?cie - logit
- aktywacja sigmoidalna `nn.Sigmoid`, przekszta?caj?ca logit na prawdopodobie?stwo klasy pozytywnej

U?yj binarnej entropii krzy?owej `nn.BCELoss` jako funkcji kosztu. U?yj optymalizatora SGD ze sta?? ucz?c? `1e-3`. Trenuj przez 3000 epok. Pami?taj, aby przekaza? do optymalizatora `torch.optim.SGD` parametry sieci (metoda `.parameters()`). Dopisz logowanie kosztu raz na 100 epok.

In [ ]:
learning_rate = 1e-3

model = nn.Linear(X_train.shape[1], 1)
activation = nn.Sigmoid()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.BCELoss()
for i in range(3000):
    # forward pass
    y_hat = activation(model(X_train))

    # loss calculation
    loss = loss_fn(y_hat, y_train)
    # backpropagation
    loss.backward()

    # optimization
    optimizer.step()
    optimizer.zero_grad()  # zeroes all gradients - very convenient!

    if i % 100 == 0:
        if loss < best_loss:
            best_model = (a.clone(), b.clone())
            best_loss = loss
        print(f"step {i} loss: {loss.item():.4f}")

print("final loss:", loss)

# implement me!
# your_code


Teraz trzeba sprawdzi?, jak posz?o naszej sieci. W PyTorchu sie? pracuje zawsze w jednym z dw?ch tryb?w: treningowym lub ewaluacyjnym (predykcyjnym). Ten drugi wy??cza niekt?re mechanizmy, kt?re s? u?ywane tylko podczas treningu, w szczeg?lno?ci regularyzacj? dropout. Do prze??czania s?u?? metody modelu `.train()` i `.eval()`.

Dodatkowo podczas liczenia predykcji dobrze jest wy??czy? liczenie gradient?w, bo nie b?d? potrzebne, a oszcz?dza to czas i pami??. U?ywa si? do tego menad?era kontekstu `with torch.no_grad():`.

In [ ]:
from sklearn.metrics import precision_recall_curve, precision_recall_fscore_support, roc_auc_score


model.eval()
with torch.no_grad():
    y_score = activation(model(X_test))

auroc = roc_auc_score(y_test, y_score)
print(f"AUROC: {auroc:.2%}")

In [ ]:
assert isinstance(model, nn.Linear)
assert isinstance(activation, nn.Sigmoid)
assert isinstance(optimizer, torch.optim.SGD)
assert isinstance(loss_fn, torch.nn.BCELoss)

assert model.out_features == 1
assert optimizer.param_groups[0]["lr"] == 1e-3

assert 0.0 < loss.item() < 0.5
assert 0.8 < auroc < 0.9

print("Solution is correct!")

Jest to ca?kiem dobry wynik, a mo?e by? jeszcze lepszy. Sprawd?my dla pewno?ci jeszcze inne metryki: precyzj?, recall oraz F1-score. Dodatkowo narysujemy krzyw? precision-recall, czyli jak zmieniaj? si? te metryki w zale?no?ci od przyj?tego progu (threshold) prawdopodobie?stwa, powy?ej kt?rego przyjmujemy klas? pozytywn?. Tak? krzyw? nale?y rysowa? na zbiorze walidacyjnym, bo p??niej chcemy wykorzysta? t? informacj? do doboru progu, a nie chcemy mie? wycieku danych testowych (data leakage).

Poni?ej zaimplementowano tak?e funkcj? `get_optimal_threshold()`, kt?ra sprawdza, dla kt?rego progu uzyskujemy maksymalny F1-score, i zwraca indeks oraz warto?? optymalnego progu. Przyda ci si? ona w dalszej cz??ci laboratorium.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay


def get_optimal_threshold(
    precisions: np.array,
    recalls: np.array,
    thresholds: np.array
) -> Tuple[int, float]:

    numerator = 2 * precisions * recalls
    denominator = precisions + recalls
    f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator != 0)

    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx]

    return optimal_idx, optimal_threshold


def plot_precision_recall_curve(y_true, y_pred_score) -> None:
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_pred_score)
    optimal_idx, optimal_threshold = get_optimal_threshold(precisions, recalls, thresholds)

    disp = PrecisionRecallDisplay(precisions, recalls)
    disp.plot()
    plt.title(f"Precision-recall curve (opt. thresh.: {optimal_threshold:.4f})")
    plt.axvline(recalls[optimal_idx], color="green", linestyle="-.")
    plt.axhline(precisions[optimal_idx], color="green", linestyle="-.")
    plt.show()



In [ ]:
model.eval()
with torch.no_grad():
    y_pred_valid_score = activation(model(X_valid))

plot_precision_recall_curve(y_valid, y_pred_valid_score)

Jak wida?, chocia? AUROC jest wysokie, to dla optymalnego F1-score recall nie jest zbyt wysoki, a precyzja jest ju? do?? niska. By? mo?e wynik uda si? poprawi?, u?ywaj?c modelu o wi?kszej pojemno?ci - pe?nej, g??bokiej sieci neuronowej.

## Sieci neuronowe

Wszystko zacz??o si? od inspirowanych biologi? [sztucznych neuron?w](https://en.wikipedia.org/wiki/Artificial_neuron), kt?rych pr?bowano u?y? do symulacji m?zgu. Naukowcy szybko odeszli od tego podej?cia (sam problem modelowania okaza? si? te? znacznie trudniejszy, ni? s?dzono), zamiast tego u?ywaj?c neuron?w jako jednostek reprezentuj?c? dowoln? funkcj? parametryczn? $f(x, \Theta)$. Ka?dy neuron jest zatem bardzo elastyczny, bo jedyne wymagania to funkcja r??niczkowalna, a mamy do tego wektor parametr?w $\Theta$.

W praktyce najcz??ciej mo?na spotka? si? z kilkoma rodzinami sieci neuronowych:
1. Perceptrony wielowarstwowe (*MultiLayer Perceptron*, MLP) - najbardziej podobne do powy?szego opisu, niezb?dne do klasyfikacji i regresji
2. Konwolucyjne (*Convolutional Neural Networks*, CNNs) - do przetwarzania danych z zale?no?ciami przestrzennymi, np. obraz?w czy d?wi?ku
3. Rekurencyjne (*Recurrent Neural Networks*, RNNs) - do przetwarzania danych z zale?no?ciami sekwencyjnymi, np. szeregi czasowe, oraz kiedy? do j?zyka naturalnego
4. Transformacyjne (*Transformers*), oparte o mechanizm atencji (*attention*) - do przetwarzania j?zyka naturalnego (NLP), z kt?rego wypar?y RNNs, a coraz cz??ciej tak?e do wszelkich innych danych, np. obraz?w, d?wi?ku
5. Grafowe (*Graph Neural Networks*, GNNS) - do przetwarzania graf?w

Na tym laboratorium skupimy si? na najprostszej architekturze, czyli MLP. Jest ona powszechnie ??czona z wszelkimi innymi architekturami, bo pozwala dokonywa? klasyfikacji i regresji. Przyk?adowo, klasyfikacja obraz?w to zwykle CNN + MLP, klasyfikacja tekst?w to transformer + MLP, a regresja na grafach to GNN + MLP.

Dodatkowo, pomimo prostoty MLP s? bardzo pot??ne - udowodniono, ?e perceptrony (ich powszechna nazwa) s? [uniwersalnym aproksymatorem](https://www.sciencedirect.com/science/article/abs/pii/0893608089900208), b?d?cym w stanie przybli?y? dowoln? funkcj? z odpowiednio ma?ym b??dem, zak?adaj?c wystarczaj?c? wielko?? warstw sieci. Szczeg?lne ich wersje potrafi? nawet [reprezentowa? drzewa decyzyjne](https://www.youtube.com/watch?v=_okxGdHM5b8).

Dla zainteresowanych polecamy [doskona?? ksi??k? "Dive into Deep Learning", z implementacjami w PyTorchu](https://d2l.ai/chapter_multilayer-perceptrons/index.html), [klasyczn? ksi??k? "Deep Learning Book"](https://www.deeplearningbook.org/contents/mlp.html), oraz [ten filmik](https://www.youtube.com/watch?v=BFHrIxKcLjA), je?li zastanawia?e?/-a? si?, czemu u?ywamy deep learning, a nie naprzyk?ad (wide?) learning. (aka. czemu staramy si? budowa? g??bokie sieci, a nie p?ytkie za to szerokie)

### Sieci MLP

Dla przypomnienia, na wej?ciu mamy punkty ze zbioru treningowego, czyli $d$-wymiarowe wektory. W klasyfikacji chcemy znale?? granic? decyzyjn?, czyli krzyw?, kt?ra oddzieli od siebie klasy. W wej?ciowej przestrzeni mo?e by? to trudne, bo chmury punkt?w z poszczeg?lnych klas mog? by? ze sob? do?? pomieszane. Pami?tajmy te?, ?e regresja logistyczna jest klasyfikatorem liniowym, czyli w danej przestrzeni potrafi oddzieli? punkty tylko lini? prost?.

Sie? MLP sk?ada si? z warstw. Ka?da z nich dokonuje nieliniowego przekszta?cenia przestrzeni (mo?na o tym my?le? jak o sk?adaniu przestrzeni jak?? prost?/?aman?), tak, aby w finalnej przestrzeni nasze punkty by?y mo?liwie liniowo separowalne. Wtedy ostatnia warstwa z sigmoid? b?dzie potrafi?a je rozdzieli? od siebie.



Poszczeg?lne neurony sk?adaj? si? z iloczynu skalarnego wej?? z wagami neuronu, oraz nieliniowej funkcji aktywacji. W PyTorchu s? to osobne obiekty - `nn.Linear` oraz np. `nn.Sigmoid`. Funkcja aktywacji przyjmuje wynik iloczynu skalarnego i przekszta?ca go, aby sprawdzi?, jak mocno reaguje neuron na dane wej?cie. Musi by? nieliniowa z dw?ch powod?w. Po pierwsze, tylko nieliniowe przekszta?cenia s? na tyle pot??ne, ?eby umo?liwi? liniow? separacj? danych w ostatniej warstwie. Po drugie, liniowe przekszta?cenia zwyczajnie nie dzia?aj?. Aby zrozumie? czemu, trzeba zobaczy?, co matematycznie oznacza sie? MLP.

![perceptron](https://www.saedsayad.com/images/Perceptron_bkp_1.png)

Zapisane matematycznie MLP to:

$\large
h_1 = f_1(x) \\
h_2 = f_2(h_1) \\
h_3 = f_3(h_2) \\
... \\
h_n = f_n(h_{n-1})
$

gdzie $x$ to wej?cie $f_i$ to funkcja aktywacji $i$-tej warstwy, a $h_i$ to wyj?cie $i$-tej warstwy, nazywane **ukryt? reprezentacj? (hidden representation)**, lub *latent representation*. Nazwa bierze si? z tego, ?e w ?rodku sieci wyci?gamy cechy i wzorce w danych, kt?re nie s? widoczne na pierwszy rzut oka na wej?ciu.

Za???my, ?e uczymy si? na danych $x$ o jednym wymiarze (dla uproszczenia wzor?w) oraz nie mamy funkcji aktywacji, czyli wykorzystujemy tak naprawd? aktywacj? liniow? $f(x) = x$. Zobaczmy jak b?d? wygl?da? dane przechodz?c przez kolejne warstwy:

$\large
h_1 = f_1(xw_1) = xw_1 \\
h_2 = f_2(h_1w_2) = xw_1w_2 \\
... \\
h_n = f_n(h_{n-1}w_n) = xw_1w_2...w_n
$

gdzie $w_i$ to jest parametr $i$-tej warstwy sieci, $x$ to s? dane (w naszym przypadku jedna liczba) wej?ciowa, a $h_i$ to wyj?cie $i$-tej warstwy.

Jak wida?, taka sie? o $n$ warstwach jest r?wnowa?na sieci o jednej warstwie z parametrem $w = w_1w_2...w_n$. Wynika to z tego, ?e z?o?enie funkcji liniowych jest tak?e funkcj? liniow? - patrz notatki z algebry :)

Je?eli natomiast u?yjemy nieliniowej funkcji aktywacji, cz?sto oznaczanej jako $\sigma$, to wszystko b?dzie dzia?a?. Co wa?ne, ostatnia warstwa, daj?ca wyj?cie sieci, ma zwykle inn? aktywacj? od warstw wewn?trz sieci, bo te? ma inne zadanie - zwr?ci? warto?? dla klasyfikacji lub regresji. Na wyj?ciu korzysta si? z funkcji liniowej (regresja), sigmoidalnej (klasyfikacja binarna) lub softmax (klasyfikacja wieloklasowa).

Wewn?trz sieci u?ywano kiedy? sigmoidy oraz tangensa hiperbolicznego `tanh`, ale okaza?o si? to nieefektywne przy uczeniu g??bokich sieci o wielu warstwach. Nowoczesne sieci korzystaj? zwykle z funkcji ReLU (*rectified linear unit*), kt?ra jest zaskakuj?co prosta: $ReLU(x) = \max(0, x)$. Okaza?o si?, ?e bardzo dobrze nadaje si? do treningu nawet bardzo g??bokich sieci neuronowych. Nowsze funkcje aktywacji s? g??wnie modyfikacjami ReLU.

![relu](https://www.nomidl.com/wp-content/uploads/2022/04/image-10.png)

### MLP w PyTorchu

Warstw? neuron?w w MLP nazywa si? warstw? g?st? (*dense layer*) lub warstw? w pe?ni po??czon? (*fully-connected layer*), i taki opis oznacza zwykle same neurony oraz funkcj? aktywacji. PyTorch, jak ju? widzieli?my, definiuje osobno transformacj? liniow? oraz aktywacj?, a wi?c jedna warstwa sk?ada si? de facto z 2 obiekt?w, wywo?ywanych jeden po drugim. Inne frameworki, szczeg?lnie wysokopoziomowe (np. Keras) ??cz? to cz?sto w jeden obiekt.

MLP sk?ada si? zatem z sekwencji obiekt?w, kt?re potem wywo?uje si? jeden po drugim, gdzie wyj?cie poprzedniego to wej?cie kolejnego. Ale nie mo?na tutaj u?ywa? Pythonowych list! Z perspektywy PyTorcha to wtedy niezale?ne obiekty i nie zostanie wtedy przekazany mi?dzy nimi gradient. Trzeba tutaj skorzysta? z `nn.Sequential`, aby tworzy? taki pipeline.

Rozmiary wej?cia i wyj?cia dla ka?dej warstwy trzeba w PyTorchu podawa? explicite. Jest to po pierwsze edukacyjne, a po drugie cz?sto u?atwia wnioskowanie o dzia?aniu sieci oraz jej debugowanie - mamy jasno podane, czego oczekujemy. Niekt?re frameworki (np. Keras) obliczaj? to automatycznie.

Co wa?ne, ostatnia warstwa zwykle nie ma funkcji aktywacji. Wynika to z tego, ?e obliczanie wielu funkcji kosztu (np. entropii krzy?owej) na aktywacjach jest cz?sto niestabilne numerycznie. Z tego powodu PyTorch oferuje funkcje kosztu zawieraj?ce w ?rodku aktywacj? dla ostatniej warstwy, a ich implementacje s? stabilne numerycznie. Przyk?adowo, `nn.BCELoss` przyjmuje wej?cie z zaaplikowanymi ju? aktywacjami, ale mo?e skutkowa? under/overflow, natomiast `nn.BCEWithLogitsLoss` przyjmuje wej?cie bez aktywacji, a w ?rodku ma specjaln? implementacj? ??cz?c? binarn? entropi? krzy?ow? z aktywacj? sigmoidaln?. Oczywi?cie w zwi?zku z tym aby dokona? potem predykcji w praktyce, trzeba pami?ta? o u?yciu funkcji aktywacji. Cz?sto korzysta si? przy tym z funkcji z modu?u `torch.nn.functional`, kt?re s? w tym wypadku nieco wygodniejsze od klas wywo?ywalnych z `torch.nn`.

Ca?e sieci w PyTorchu tworzy si? jako klasy dziedzicz?ce po `nn.Module`. Co wa?ne, obiekty, z kt?rych tworzymy sie?, np. `nn.Linear`, tak?e dziedzicz? po tej klasie. Pozwala to na bardzo modu?ow? budow? kodu, zgodn? z zasadami OOP. W konstruktorze najpierw trzeba zawsze wywo?a? konstruktor rodzica - `super().__init__()`, a p??niej tworzy si? potrzebne obiekty i zapisuje jako atrybuty. Ka?dy atrybut dziedzicz?cy po `nn.Module` lub `nn.Parameter` jest uwa?any za taki, kt?ry zawiera parametry sieci, a wi?c przy wywo?aniu metody `parameters()` - parametry z tych atrybut?w pojawi? si? w li?cie wszystkich parametr?w. Musimy te? zdefiniowa? metod? `forward()`, kt?ra przyjmuje tensor `x` i zwraca wynik. Typowo ta metoda po prostu u?ywa obiekt?w zdefiniowanych w konstruktorze.


**UWAGA: nigdy w normalnych warunkach si? nie wo?a metody `forward` r?cznie**

### Zadanie 4 (0.5 punktu)

Uzupe?nij implementacj? 3-warstwowej sieci MLP. U?yj rozmiar?w:
* pierwsza warstwa: input_size x 256
* druga warstwa: 256 x 128
* trzecia warstwa: 128 x 1

U?yj funkcji aktywacji ReLU.

Przydatne klasy:
- `nn.Sequential`
- `nn.Linear`
- `nn.ReLU`

In [ ]:
from torch import sigmoid


class MLP(nn.Module):
    def __init__(self, input_size: int):
        super().__init__()
        self.model=nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
        )
        # implement me!
        # your_code

    def forward(self, x):
        # implement me!
        # your_code
        return self.model(x)

    def predict_proba(self, x):
        return sigmoid(self(x))

    def predict(self, x, threshold: float = 0.5):
        y_pred_score = self.predict_proba(x)
        return (y_pred_score > threshold).to(torch.int32)



In [ ]:
# 43s
learning_rate = 1e-3
model = MLP(input_size=X_train.shape[1])
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# note that we are using loss function with sigmoid built in
loss_fn = torch.nn.BCEWithLogitsLoss()
num_epochs = 2000
evaluation_steps = 200

for i in range(num_epochs):
    y_pred = model(X_train)
    loss = loss_fn(y_pred, y_train)
    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if i % evaluation_steps == 0:
        print(f"Epoch {i} train loss: {loss.item():.4f}")

print(f"final loss: {loss.item():.4f}")

In [ ]:
model.eval()
with torch.no_grad():
    # positive class probabilities
    y_pred_valid_score = model.predict_proba(X_valid)
    y_pred_test_score = model.predict_proba(X_test)

auroc = roc_auc_score(y_test, y_pred_test_score)
print(f"AUROC: {auroc:.2%}")

plot_precision_recall_curve(y_valid, y_pred_valid_score)

In [ ]:
assert any(
    isinstance(module, nn.Linear) and module.in_features == X_train.shape[1] and module.out_features == 256
    for module in model.modules()
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == 256 and module.out_features == 128
    for module in model.modules()
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == 128 and module.out_features == 1
    for module in model.modules()
)

assert any(isinstance(module, nn.ReLU) for module in model.modules())

assert 0.5 < loss.item() < 0.6
assert 0.6 < auroc < 0.9

print("Solution is correct!")

AUROC jest podobne, a precision i recall spad?y - wypadamy wr?cz gorzej od regresji liniowej! Skoro dodali?my wi?cej warstw, to mo?e pojemno?? modelu jest teraz za du?a i trzeba by go zregularyzowa??

Sieci neuronowe bardzo ?atwo przeuczaj?, bo s? bardzo elastycznymi i pojemnymi modelami. Dlatego maj? wiele r??nych rodzaj?w regularyzacji, kt?rych u?ywa si? razem. Co ciekawe, udowodniono eksperymentalnie, ?e zbyt du?e sieci z mocn? regularyzacj? dzia?aj? lepiej ni? mniejsze sieci, odpowiedniego rozmiaru, za to ze s?absz? regularyzacj?.

Pierwszy rodzaj regularyzacji to znana nam ju? **regularyzacja L2**, czyli penalizacja zbyt du?ych wag. W kontek?cie sieci neuronowych nazywa si? te? j? czasem *weight decay*. W PyTorchu dodaje si? j? jako argument do optymalizatora.

Regularyzacja specyficzna dla sieci neuronowych to **dropout**. Polega on na losowym wy??czaniu zadanego procenta neuron?w podczas treningu. Pomimo prostoty okaza?a si? niesamowicie skuteczna, szczeg?lnie w treningu bardzo g??bokich sieci. Co wa?ne, jest to mechanizm u?ywany tylko podczas treningu - w trakcie predykcji za pomoc? sieci wy??cza si? ten mechanizm i dokonuje normalnie predykcji ca?? sieci?. Podej?cie to mo?na potraktowa? jak ensemble learning, podobny do las?w losowych - wy??czaj?c losowe cz??ci sieci, w ka?dej iteracji trenujemy nieco inn? sie?, co odpowiada u?rednianiu predykcji r??nych algorytm?w. Typowo stosuje si? do?? mocny dropout, rz?du 25-50%. W PyTorchu implementuje go warstwa `nn.Dropout`, aplikowana zazwyczaj po funkcji aktywacji.

Ostatni, a by? mo?e najwa?niejszy rodzaj regularyzacji to **wczesny stop (early stopping)**. W ka?dym kroku mocniej dostosowujemy terenow? sie? do zbioru treningowego, a wi?c zbyt d?ugi trening b?dzie skutkowa? przeuczeniem. W metodzie wczesnego stopu u?ywamy wydzielonego zbioru walidacyjnego (pojedynczego, metoda holdout), sprawdzaj?c co okre?lon? liczb? epok wynik na tym zbiorze. Je?eli nie uzyskamy wyniku lepszego od najlepszego dotychczas uzyskanego przez okre?lon? liczb? epok, to przerywamy trening. Okres, przez kt?ry czekamy na uzyskanie lepszego wyniku, to cierpliwo?? (*patience*). Im mniejsze, tym mocniejszy jest ten rodzaj regularyzacji, ale trzeba z tym uwa?a?, bo ?atwo jest przesadzi? i zbyt szybko przerywa? trening. Niekt?re implementacje uwzgl?dniaj? tzw. *grace period*, czyli gwarantowan? minimaln? liczb? epok, przez kt?r? b?dziemy trenowa? sie?, niezale?nie od wybranej cierpliwo?ci.

Dodatkowo ryzyko przeuczenia mo?na zmniejszy?, u?ywaj?c mniejszej sta?ej ucz?cej.

### Zadanie 5 (1.5 punktu)

Zaimplementuj funkcj? `evaluate_model()`, obliczaj?c? metryki na zbiorze testowym:
- warto?? funkcji kosztu (loss)
- AUROC
- optymalny pr?g
- F1-score przy optymalnym progu
- precyzj? oraz recall dla optymalnego progu

Je?eli podana jest warto?? argumentu `threshold`, to u?yj jej do zamiany prawdopodobie?stw na twarde predykcje. W przeciwnym razie u?yj funkcji `get_optimal_threshold` i oblicz optymaln? warto?? progu.

Pami?taj o prze??czeniu modelu w tryb ewaluacji oraz o wy??czeniu obliczania gradient?w.

In [ ]:
from typing import Optional

from sklearn.metrics import precision_score, recall_score, f1_score
from torch import sigmoid


def evaluate_model(
    model: nn.Module,
    X: torch.Tensor,
    y: torch.Tensor,
    loss_fn: nn.Module,
    threshold: Optional[float] = None
) -> Dict[str, float]:
    # implement me!
    # your_code
    model.eval()
    with torch.no_grad():
      y_pred_proba = model(X)
      loss = loss_fn(y_pred_proba, y)
      auroc = roc_auc_score(y, y_pred_proba)
      if threshold is None:
            precisions, recalls, thresholds = precision_recall_curve(y, y_pred_proba)
            _, threshold = get_optimal_threshold(precisions, recalls, thresholds)
      y_pred = (y_pred_proba > threshold).to(torch.int32)
      precisions = precision_score(y, y_pred)
      recalls = recall_score(y, y_pred)
      f1 = f1_score(y, y_pred)



    return {
        "loss": loss.item(),
        "AUROC": auroc,
        "precision": precisions,
        "recall": recalls,
        "F1-score": f1,
        "threshold": threshold,

    }



In [ ]:
eval_result = evaluate_model(model, X_train, y_train, loss_fn)


assert 0.6 < eval_result["AUROC"] < 0.9
assert 0.3 < eval_result["precision"] < 0.6
assert 0.6 < eval_result["recall"] < 0.9
assert 0.4 < eval_result["F1-score"] < 0.7
assert 0.5 < eval_result["loss"] < 0.6
print("Solution is correct!")

### Zadanie 6 (0.5 punktu)

Zaimplementuj 3-warstwow? sie? MLP z dropout (50%). Rozmiary warstw ukrytych maj? wynosi? 256 i 128.

In [ ]:
class RegularizedMLP(nn.Module):
    def __init__(self, input_size: int, dropout_p: float = 0.5):
        super().__init__()
        self.rmlp=nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 1),
        )
        # implement me!
        # your_code

    def forward(self, x):
        # implement me!
        # your_code
        return self.rmlp(x)

    def predict_proba(self, x):
        return sigmoid(self(x))

    def predict(self, x, threshold: float = 0.5):
        y_pred_score = self.predict_proba(x)
        return (y_pred_score > threshold).to(torch.int32)



In [ ]:
verification_input_size = 143
verification_dropout_p = 0.125
verification_model = RegularizedMLP(
    input_size=verification_input_size,
    dropout_p=verification_dropout_p,
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == verification_input_size and module.out_features == 256
    for module in verification_model.modules()
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == 256 and module.out_features == 128
    for module in verification_model.modules()
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == 128 and module.out_features == 1
    for module in verification_model.modules()
)

assert any(isinstance(module, nn.Dropout) and module.p == verification_dropout_p for module in verification_model.modules())
assert any(isinstance(module, nn.ReLU) for module in verification_model.modules())

print("Solution is correct!")

Opisali?my wcze?niej podstawowy optymalizator w sieciach neuronowych - spadek wzd?u? gradientu. Jednak wymaga on u?ycia ca?ego zbioru danych, aby obliczy? gradient, co jest cz?sto niewykonalne przez rozmiar zbioru. Dlatego wymy?lono **stochastyczny spadek wzd?u? gradientu (stochastic gradient descent, SGD)**, w kt?rym u?ywamy 1 przyk?adu naraz, liczymy gradient tylko po nim i aktualizujemy parametry. Jest to oczywi?cie do?? grube przybli?enie gradientu, ale pozwala robi? szybko du?o ma?ych krok?w. Kompromisem, kt?rego u?ywa si? w praktyce, jest **minibatch gradient descent**, czyli u?ywanie batchy np. 32, 64 czy 128 przyk?ad?w.

Rzadko wspominanym, a wa?nym faktem jest tak?e to, ?e stochastyczno?? metody optymalizacji jest sama w sobie te? [metod? regularyzacji](https://arxiv.org/abs/2101.12176), a wi?c `batch_size` to tak?e hiperparametr.

Obecnie najpopularniejsz? odmian? SGD jest [Adam](https://arxiv.org/abs/1412.6980), gdy? uczy on szybko sie? oraz daje bardzo dobre wyniki nawet przy niekoniecznie idealnie dobranych hiperparametrach. W PyTorchu najlepiej korzysta? z jego implementacji `AdamW`, kt?ra jest nieco lepsza ni? implementacja `Adam`. Jest to zasadniczo zawsze wyb?r domy?lny przy treningu wsp??czesnych sieci neuronowych.

Na razie u?yjemy jednak minibatch SGD.

Poni?ej znajduje si? implementacja prostej klasy dziedzicz?cej po `Dataset` - tak w PyTorchu implementuje si? w?asne zbiory danych. U?ycie takich klas umo?liwia u?ycie klas ?aduj?cych dane (`DataLoader`), kt?re z kolei pozwalaj? ?atwo ?adowa? batche danych. Trzeba w takiej klasie zaimplementowa? metody:
- `__len__` - zwraca ilo?? punkt?w w zbiorze
- `__getitem__` - zwraca przyk?ad ze zbioru pod danym indeksem oraz jego klas?


In [ ]:
from torch.utils.data import Dataset


class MyDataset(Dataset):
    def __init__(self, data, y):
        super().__init__()

        self.data = data
        self.y = y

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], self.y[idx]



### Zadanie 7 (1.5 punktu)

Zaimplementuj p?tl? treningowo-walidacyjn? dla sieci neuronowej. Wykorzystaj podane warto?ci hiperparametr?w do treningu (sta?a ucz?ca, prawdopodobie?stwo dropoutu, regularyzacja L2, rozmiar batcha, maksymalna liczba epok). U?yj optymalizatora SGD.

Dodatkowo zaimplementuj regularyzacj? przez early stopping. Sprawdzaj co epok? wynik na zbiorze walidacyjnym. U?yj podanej warto?ci patience, a jako metryki po prostu warto?ci funkcji kosztu. Mo?e si? tutaj przyda? zaimplementowana funkcja `evaluate_model()`.

Pami?taj o tym, aby przechowywa? najlepszy dotychczasowy wynik walidacyjny oraz najlepszy dotychczasowy model. Zapami?taj te? optymalny pr?g do klasyfikacji dla najlepszego modelu.

In [ ]:
from copy import deepcopy

from torch.utils.data import DataLoader


learning_rate = 1e-3
dropout_p = 0.5
l2_reg = 1e-4
batch_size = 128
max_epochs = 300

early_stopping_patience = 4

In [ ]:
# 1m
model = RegularizedMLP(
    input_size=X_train.shape[1],
    dropout_p=dropout_p
)
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=learning_rate,
    weight_decay=l2_reg
)
loss_fn = torch.nn.BCEWithLogitsLoss()

train_dataset = MyDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

steps_without_improvement = 0

best_val_loss = np.inf
best_model = None
best_threshold = None

for epoch_num in range(max_epochs):
    model.train()

    # note that we are using DataLoader to get batches
    for X_batch, y_batch in train_dataloader:
      y_pred = model(X_batch)
      loss = loss_fn(y_pred, y_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

        # model training
        # implement me!
        # your_code

    # model evaluation, early stopping
    # implement me!
    # your_code
    valid_metrics = evaluate_model(model, X_valid, y_valid, loss_fn , best_threshold)
    if valid_metrics["loss"] < best_val_loss:
        best_val_loss = valid_metrics["loss"]
        best_model = deepcopy(model)
        best_threshold = valid_metrics["threshold"]
        steps_without_improvement = 0
    else:
        steps_without_improvement += 1
        if steps_without_improvement >= early_stopping_patience:
            break
    print(f"Epoch {epoch_num} train loss: {loss.item():.4f}, eval loss {valid_metrics['loss']:.4f}")

In [ ]:
test_metrics = evaluate_model(best_model, X_test, y_test, loss_fn, best_threshold)

print(f"AUROC: {test_metrics['AUROC']:.2%}")
print(f"F1: {test_metrics['F1-score']:.2%}")
print(f"Precision: {test_metrics['precision']:.2%}")
print(f"Recall: {test_metrics['recall']:.2%}")

In [ ]:
assert test_metrics["AUROC"] > 0.8
assert test_metrics["F1-score"] > 0.6
assert test_metrics["precision"] > 0.5
assert test_metrics["recall"] > 0.6

print("Solution is correct!")

Wyniki wygl?daj? ju? du?o lepiej.

Na koniec laboratorium do?o?ymy do naszego modelu jeszcze 3 powrzechnie u?ywane techniki, kt?re s? bardzo proste, a pozwalaj? cz?sto ulepszy? wynik modelu.

Pierwsz? z nich s? **warstwy normalizacji (normalization layers)**. Powsta?y one pocz?tkowo z za?o?eniem, ?e przez przekszta?cenia przestrzeni dokonywane przez sie? zmienia si? rozk?ad prawdopodobie?stw pomi?dzy warstwami, czyli tzw. *internal covariate shift*. P??niej okaza?o si?, ?e zastosowanie takiej normalizacji wyg?adza powierzchni? funkcji kosztu, co u?atwia i przyspiesza optymalizacj?. Najpowszechniej u?ywan? normalizacj? jest **batch normalization (batch norm)**.

Drugim ulepszeniem jest dodanie **wag klas (class weights)**. Mamy do czynienia z problemem klasyfikacji niezbalansowanej, wi?c klasa mniejszo?ciowa, wa?niejsza dla nas, powinna dosta? wi?ksz? wag?. Implementuje si? to trywialnie prosto - po prostu mno?ymy warto?? funkcji kosztu dla danego przyk?adu przez wag? dla prawdziwej klasy tego przyk?adu. Praktycznie ka?dy klasyfikator operuj?cy na jakiej? wa?onej funkcji mo?e dzia?a? w ten spos?b, nie tylko sieci neuronowe.

Ostatnim ulepszeniem jest zamiana SGD na optymalizator Adam, a konkretnie na optymalizator `AdamW`. Jest to przyk?ad **optymalizatora adaptacyjnego (adaptive optimizer)**, kt?ry potrafi zaadaptowa? sta?? ucz?c? dla ka?dego parametru z osobna w trakcie treningu. Wykorzystuje do tego gradienty - w uproszczeniu, im wi?ksza wariancja gradientu, tym mniejsze kroki w tym kierunku robimy.

### Zadanie 8 (0.5 punktu)

Zaimplementuj model `NormalizingMLP`, o takiej samej strukturze jak `RegularizedMLP`, ale dodatkowo z warstwami `BatchNorm1d` pomi?dzy warstwami `Linear` oraz `ReLU`.

Za pomoc? funkcji `compute_class_weight()` oblicz wagi dla poszczeg?lnych klas. U?yj opcji `"balanced"`. Przeka? do funkcji kosztu wag? klasy pozytywnej (pami?taj, aby zamieni? j? na tensor).

Zamie? u?ywany optymalizator na `AdamW`.

Na koniec skopiuj reszt? kodu do treningu z poprzedniego zadania, wytrenuj sie? i oblicz wyniki na zbiorze testowym.

In [ ]:
class NormalizingMLP(nn.Module):
    def __init__(self, input_size: int, dropout_p: float = 0.5):
        super().__init__()

        # implement me!
        # your_code
        self.mlp=nn.Sequential(
            nn.Linear(input_size, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.mlp(x)

    def predict_proba(self, x):
        return sigmoid(self(x))

    def predict(self, x, threshold: float = 0.5):
        y_pred_score = self.predict_proba(x)
        return (y_pred_score > threshold).to(torch.int32)

In [ ]:
# define all the hyperparameters
# your_code
learning_rate = 1e-3
dropout_p = 0.5
l2_reg = 1e-4
batch_size = 128
max_epochs = 300

early_stopping_patience = 4

In [ ]:
# training loop
# your_code
# 1m
from sklearn.utils.class_weight import compute_class_weight
weights = compute_class_weight("balanced", classes=np.unique(y_train.squeeze().numpy()), y= y_train.squeeze().numpy())
pos_weights = torch.tensor(weights[1], dtype=torch.float)
model = NormalizingMLP(
    input_size=X_train.shape[1],
    dropout_p=dropout_p
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=l2_reg
)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)

train_dataset = MyDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

steps_without_improvement = 0

best_val_loss = np.inf
best_model = None
best_threshold = None

for epoch_num in range(max_epochs):
    model.train()

    # note that we are using DataLoader to get batches
    for X_batch, y_batch in train_dataloader:
      y_pred = model(X_batch)
      loss = loss_fn(y_pred, y_batch)
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

        # model training
        # implement me!
        # your_code

    # model evaluation, early stopping
    # implement me!
    # your_code
    valid_metrics = evaluate_model(model, X_valid, y_valid, loss_fn , best_threshold)
    if valid_metrics["loss"] < best_val_loss:
        best_val_loss = valid_metrics["loss"]
        best_model = deepcopy(model)
        best_threshold = valid_metrics["threshold"]
        steps_without_improvement = 0
    else:
        steps_without_improvement += 1
        if steps_without_improvement >= early_stopping_patience:
            break
    print(f"Epoch {epoch_num} train loss: {loss.item():.4f}, eval loss {valid_metrics['loss']:.4f}")

In [ ]:
test_metrics = evaluate_model(best_model, X_test, y_test, loss_fn, best_threshold)

print(f"AUROC: {test_metrics['AUROC']:.2%}")
print(f"F1: {test_metrics['F1-score']:.2%}")
print(f"Precision: {test_metrics['precision']:.2%}")
print(f"Recall: {test_metrics['recall']:.2%}")

In [ ]:
verification_input_size = 143
verification_dropout_p = 0.125
verification_model = NormalizingMLP(
    input_size=verification_input_size,
    dropout_p=verification_dropout_p,
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == verification_input_size and module.out_features == 256
    for module in verification_model.modules()
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == 256 and module.out_features == 128
    for module in verification_model.modules()
)

assert any(
    isinstance(module, nn.Linear) and module.in_features == 128 and module.out_features == 1
    for module in verification_model.modules()
)

assert any(isinstance(module, nn.Dropout) and module.p == verification_dropout_p for module in verification_model.modules())
assert any(isinstance(module, nn.ReLU) for module in verification_model.modules())

assert any(isinstance(module, nn.BatchNorm1d) for module in verification_model.modules())

assert test_metrics["AUROC"] > 0.8
assert test_metrics["F1-score"] > 0.6
assert test_metrics["precision"] > 0.5
assert test_metrics["recall"] > 0.6

print("Solution is correct!")

## Akceleracja sprz?towa (dla zainteresowanych)

Jak wcze?niej wspominali?my, u?ycie akceleracji sprz?towej, czyli po prostu GPU do oblicze?, jest bardzo efektywne w przypadku sieci neuronowych. Karty graficzne bardzo efektywnie mno?? macierze, a sieci neuronowe to, jak mo?na by?o si? przekona?, du?o mno?enia macierzy.

W PyTorchu jest to dosy? ?atwe, ale trzeba robi? to explicite. S?u?y do tego metoda `.to()`, kt?ra przenosi tensory mi?dzy CPU i GPU. Poni?ej przyk?ad, jak to si? robi (oczywi?cie trzeba mie? skonfigurowane GPU, ?eby dzia?a?o):

In [ ]:
import time


class CudaMLP(nn.Module):
    def __init__(self, input_size: int, dropout_p: float = 0.5):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.mlp(x)

    def predict_proba(self, x):
        return sigmoid(self(x))

    def predict(self, x, threshold: float = 0.5):
        y_pred_score = self.predict_proba(x)
        return (y_pred_score > threshold).to(torch.int32)


model = CudaMLP(X_train.shape[1]).to('cuda')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

# note that we are using loss function with sigmoid built in
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.from_numpy(weights)[1].to('cuda'))

step_counter = 0
time_from_eval = time.time()
for epoch_id in range(30):
    for batch_x, batch_y in train_dataloader:
        batch_x = batch_x.to('cuda')
        batch_y = batch_y.to('cuda')

        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        if step_counter % evaluation_steps == 0:
            print(f"Epoch {epoch_id} train loss: {loss.item():.4f}, time: {time.time() - time_from_eval}")
            time_from_eval = time.time()

        step_counter += 1

test_res = evaluate_model(model.to('cpu'), X_test, y_test, loss_fn.to('cpu'), threshold=0.5)

print(f"AUROC: {test_res['AUROC']:.2%}")
print(f"F1: {test_res['F1-score']:.2%}")
print(test_res)

Co prawda ten model nie b?dzie tak dobry jak ten z laboratorium, ale zwr?? uwag?, o ile jest wi?kszy, a przy tym szybszy.

Dla zainteresowanych polecamy [t? serie artyku??w](https://medium.com/@adi.fu7/ai-accelerators-part-i-intro-822c2cdb4ca4)

## Zadanie dla ch?tnych

Jak widzieli?my, sieci neuronowe maj? bardzo du?o hiperparametr?w. Przeszukiwanie ich grid search'em jest wi?c niewykonalne, a chocia? random search by dzia?a?, to potrzebowa?by wielu iteracji, co te? jest kosztowne obliczeniowo.

Zaimplementuj inteligentne przeszukiwanie przestrzeni hiperparametr?w za pomoc? biblioteki [Optuna](https://optuna.org/). Implementuje ona mi?dzy innymi algorytm Tree Parzen Estimator (TPE), nale??cy do grupy algorytm?w typu Bayesian search. Typowo osi?gaj? one bardzo dobre wyniki, a w?a?ciwie zawsze lepsze od przeszukiwania losowego. Do tego wystarcza im cz?sto niewielka liczba krok?w.

Zaimplementuj 3-warstwow? sie? MLP, gdzie pierwsza warstwa ma rozmiar ukryty N, a druga N // 2. Ucz j? optymalizatorem Adam przez maksymalnie 300 epok z cierpliwo?ci? 10.

Przeszukaj wybrane zakresy dla hiperparametr?w:
- rozmiar warstw ukrytych (N)
- sta?a ucz?ca
- batch size
- si?a regularyzacji L2
- prawdopodobie?stwo dropoutu

Wykorzystaj przynajmniej 30 iteracji. Nast?pnie prze??cz algorytm na losowy (Optuna tak?e jego implementuje), wykonaj 30 iteracji i por?wnaj jako?? wynik?w.

Przydatne materia?y:
- [Optuna code examples - PyTorch](https://optuna.org/#code_examples)
- [Auto-Tuning Hyperparameters with Optuna and PyTorch](https://www.youtube.com/watch?v=P6NwZVl8ttc)
- [Hyperparameter Tuning of Neural Networks with Optuna and PyTorch](https://towardsdatascience.com/hyperparameter-tuning-of-neural-networks-with-optuna-and-pytorch-22e179efc837)
- [Using Optuna to Optimize PyTorch Hyperparameters](https://medium.com/pytorch/using-optuna-to-optimize-pytorch-hyperparameters-990607385e36)

In [ ]:
learning_rate = 1e-3
dropout_p = 0.5
l2_reg = 1e-4
batch_size = 128
max_epochs = 300
early_stopping_patience = 10
weights = compute_class_weight("balanced", classes=np.unique(y_train.squeeze().numpy()), y= y_train.squeeze().numpy())
pos_weights = torch.tensor(weights[1], dtype=torch.float)

In [ ]:
class NewMLP(nn.Module):
    def __init__(self, input_size: int, n_leyers: int, dropout_p: float = 0.5):
        super().__init__()

        # implement me!
        # your_code
        self.mlp=nn.Sequential(
            nn.Linear(input_size, n_leyers),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(n_leyers, n_leyers//2),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(n_leyers//2, 1)

        )

    def forward(self, x):
        return self.mlp(x)

    def predict_proba(self, x):
        return sigmoid(self(x))

    def predict(self, x, threshold: float = 0.5):
        y_pred_score = self.predict_proba(x)
        return (y_pred_score > threshold).to(torch.int32)

In [ ]:
!pip install optuna

In [ ]:
import optuna
def objective(trial):
  n_leyers = trial.suggest_int("n_leyers", 128, 512)
  learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
  batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128, 256])
  l2_reg = trial.suggest_float("l2_reg", 1e-5, 1e-1, log=True)
  dropout_p = trial.suggest_float("dropout_p", 0.1, 0.5)
  model = NewMLP(
      input_size=X_train.shape[1],
      n_leyers=n_leyers,
      dropout_p=dropout_p

  )
  return train_model(learning_rate, batch_size, l2_reg, model)

def train_model( learning_rate, batch_size, l2_reg, model):
    optimizer = torch.optim.AdamW(
      model.parameters(),
      lr=learning_rate,
      weight_decay=l2_reg
  )
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)

    train_dataset = MyDataset(X_train, y_train)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size)

    steps_without_improvement = 0

    best_val_loss = np.inf
    best_model = None
    best_threshold = None

    for epoch_num in range(max_epochs):
        model.train()

      # note that we are using DataLoader to get batches
        for X_batch, y_batch in train_dataloader:
          y_pred = model(X_batch)
          loss = loss_fn(y_pred, y_batch)
          loss.backward()
          optimizer.step()
          optimizer.zero_grad()
        valid_metrics = evaluate_model(model, X_valid, y_valid, loss_fn , best_threshold)
        if valid_metrics["loss"] < best_val_loss:
            best_val_loss = valid_metrics["loss"]
            best_model = deepcopy(model)
            best_threshold = valid_metrics["threshold"]
            steps_without_improvement = 0
        else:
            steps_without_improvement += 1
            if steps_without_improvement >= early_stopping_patience:
                break
    model = deepcopy(best_model)
    return best_val_loss


In [ ]:
# TPE search
study_tpe = optuna.create_study(direction="minimize")
study_tpe.optimize(objective, n_trials=30)

# Random search
study_random = optuna.create_study(direction="minimize", sampler=optuna.samplers.RandomSampler())
study_random.optimize(objective, n_trials=30)


In [ ]:
print("Best trial TPE:")
print(f"Value: {study_tpe.best_value}")
print(f"Params: {study_tpe.best_params}")

print("\nBest trial Random Search:")
print(f"Value: {study_random.best_value}")
print(f"Params: {study_random.best_params}")

In [ ]:
best_TPE_model = NewMLP(
      input_size=X_train.shape[1],
      n_leyers=study_tpe.best_params['n_leyers'],
      dropout_p=study_tpe.best_params['dropout_p']
  )
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
train_model(study_tpe.best_params['learning_rate'],study_tpe.best_params['batch_size'],study_tpe.best_params['l2_reg'],best_TPE_model)
print(evaluate_model(best_TPE_model, X_test, y_test, loss_fn))
best_Random_model = NewMLP(
      input_size=X_train.shape[1],
      n_leyers=study_random.best_params['n_leyers'],
      dropout_p=study_random.best_params['dropout_p']
  )
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weights)
train_model(study_random.best_params['learning_rate'],study_random.best_params['batch_size'],study_random.best_params['l2_reg'],best_Random_model)
print(evaluate_model(best_Random_model, X_test, y_test, loss_fn))


Obydwie metody d?y podobne wyniki aczkolwiek Random by? troch? lepszy
